In [ ]:
# Configura o Matplotlib para exibir gráficos inline no Jupyter
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import warnings

# Suprimir avisos futuros do OneHotEncoder (opcional, para limpar a saída)
warnings.filterwarnings('ignore', category=FutureWarning)

file_name = "dataset_tratado.csv"

try:
    print(f"Carregando o arquivo: {file_name}...")
    df = pd.read_csv(file_name)

    # Tratamento de dados
    clean_df = df.copy()

    # Preparação Adicional e Criação do Alvo
    print("Preparando dados e calculando 'price_per_sqm'...")
    
    clean_df['price'] = pd.to_numeric(clean_df['price'], errors='coerce')
    clean_df['baths'] = pd.to_numeric(clean_df['baths'], errors='coerce')
    clean_df['bedrooms'] = pd.to_numeric(clean_df['bedrooms'], errors='coerce')

    ml_df = clean_df.copy()
    
    # Limpar Nulos e Infinitos
    ml_df = ml_df.dropna(subset=['price', 'property_type', 'city', 'location', 'baths', 'bedrooms', 'area', 'price_per_sqm'])
    ml_df = ml_df[~np.isinf(ml_df['price_per_sqm'])]
    
    city_counts = ml_df['city'].value_counts()
    
    cities = city_counts.index.tolist()
    
    cities_to_analyze = cities 
    
    print(f"Iniciando análise para as próximas cidades mais relevantes: {cities_to_analyze}\n")
    
    # Loop de Análise para cada Cidade
    for city in cities_to_analyze:
        
        print("\n" + "="*60)
        print(f"INICIANDO ANÁLISE PARA: {city.upper()}")
        print("="*60)
        
        df_city = ml_df[ml_df['city'] == city].copy()
        
        # Checagem de segurança (ex: precisa de dados suficientes para treinar)
        if len(df_city) < 50: # Limite mínimo de 50 anúncios
            print(f"Pulando {city}, dados insuficientes (menos de 50 registros).")
            continue
            
        # --- Definir X e y 
        y = df_city['price_per_sqm']
        X = df_city.drop(['price', 'price_per_sqm', 'city', 'purpose'], axis=1)

        # --- Definir Features
        numeric_features = ['baths', 'bedrooms', 'area']
        categorical_features = ['property_type', 'location'] 
        
        # --- Criar Pipeline (Precisa ser recriado a cada loop) 
        # O OneHotEncoder precisa ser "fitado" (ajustado) para as 'locations'
        # específicas de cada nova cidade.
        preprocessor = ColumnTransformer(
            transformers=[
                ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
            ],
            remainder='passthrough'
        )

        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
        ])

        # --- Treinar Modelo
        print(f"Treinando o modelo para '{city}'...")
        pipeline.fit(X, y)
        print("Modelo treinado com sucesso.")

        # --- Extrair e Agrupar Importâncias
        print("Extraindo importância dos atributos...")
        
        encoded_cat_features = pipeline.named_steps['preprocessor'] \
                                       .named_transformers_['cat'] \
                                       .get_feature_names_out(categorical_features)
        
        numeric_features_out = [col for col in X.columns if col not in categorical_features]
        all_feature_names = np.concatenate([encoded_cat_features, numeric_features_out])
        importances = pipeline.named_steps['regressor'].feature_importances_

        feature_importance_df = pd.DataFrame({
            'Feature': all_feature_names,
            'Importance': importances
        })
        
        grouped_importances = {}
        for cat_feature in categorical_features:
            related_features = [f for f in all_feature_names if f.startswith(f"{cat_feature}_")]
            total_importance = feature_importance_df[feature_importance_df['Feature'].isin(related_features)]['Importance'].sum()
            grouped_importances[cat_feature] = total_importance

        for num_feature in numeric_features:
            total_importance = feature_importance_df[feature_importance_df['Feature'] == num_feature]['Importance'].sum()
            grouped_importances[num_feature] = total_importance
            
        grouped_importance_series = pd.Series(grouped_importances).sort_values(ascending=False)

        print(f"\n--- Importância Agrupada (Preço/m²) em {city} ---")
        print(grouped_importance_series)
        print("-------------------------------------------------\n")

        # Plotar Gráfico
        plt.figure(figsize=(10, 6))
        grouped_importance_series.sort_values(ascending=True).plot(kind='barh', color='indigo')
        
        plt.xlabel("Importância (Score Agrupado)", fontsize=12)
        plt.ylabel("Atributo", fontsize=12)
        plt.title(f"Importância dos Atributos (Preço/m²) na cidade de {city}", fontsize=16)
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.tight_layout()
        
        # Salva o arquivo com um nome único
        plt.savefig(f"importancia_m2_{city.lower().replace(' ', '_')}.png")
        print(f"Gráfico salvo como: 'importancia_m2_{city.lower().replace(' ', '_')}.png'")
        plt.show()

except FileNotFoundError:
    print(f"Erro: O arquivo '{file_name}' não foi encontrado.")
except KeyError as e:
    print(f"Erro: Coluna não encontrada: {e}. Verifique se as colunas (ex: 'Area_in_Marla', 'bedrooms') existem no seu CSV.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")